In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkConf

conf = SparkConf()
conf.setAppName("MySparkApp")
conf.set("spark.sql.files.maxPartitionBytes", "16m")
conf.setMaster("local[*]") 

spark = SparkSession.builder.config(conf = conf).getOrCreate()

In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType
schema_cat = StructType([
    StructField('session_id', IntegerType(), False),
    StructField('cat_level1', IntegerType(), False),
    StructField('cat_level2', IntegerType(), False),
    StructField('cat_level3', IntegerType(), False)
    ]
)

schema_cust = StructType([
    StructField('customer_id', IntegerType(), False),
    StructField('first_name', StringType(), True),
    StructField('last_name', StringType(), True),
    StructField('email', StringType(), True),
    StructField('gender', StringType(), True),
    StructField('birth_date', DateType(), True),
    StructField('first_join_date', DateType(), True)

])

schema_prod = StructType([
    StructField('id', IntegerType(), False),
    StructField('gender', StringType(), True),
    StructField('baseColour', StringType(), True),
    StructField('season', StringType(), True),
    StructField('year', IntegerType(), True),
    StructField('usage', StringType(), True),
    StructField('productDisplayName', StringType(), True),
    StructField('category_id', IntegerType(), True)
])

In [3]:
schema_beh = StructType([
    StructField('session_id', StringType(), False),
    StructField('event_type', StringType(), True),
    StructField('event_time', TimestampType(), True),
    StructField('traffic_source', StringType(), True),
    StructField('device_type', StringType(), True)
])

schema_trans = StructType([
    StructField('created_at', DateType(), True),
    StructField('customer_id', IntegerType(), False),
    StructField('transaction_id', IntegerType(), False),
    StructField('session_id', IntegerType(), False),
    StructField('product_metadata', StringType(), True),
    StructField('payment_method', StringType(), True),
    StructField('payment_status', StringType(), True),
    StructField('promo_amount', IntegerType(), True),
    StructField('promo_code', StringType(), True),
    StructField('shipment_fee', IntegerType(), True),
    StructField('shipment_location_lat', StringType(), True),
    StructField('shipment_location_long', StringType(), True),
    StructField('total_amount', IntegerType(), True),
    StructField('clear_payment', StringType(), True),

])

In [4]:
customer_df = spark.read.csv('./dataset/customer.csv', header=True, schema = schema_cust)
category_df = spark.read.csv('./dataset/category.csv', header=True, schema = schema_cat)
product_df = spark.read.csv('./dataset/product.csv', header=True, schema = schema_prod)
b_behavior_df = spark.read.csv('./dataset/browsing_behaviour.csv', header=True, schema = schema_beh)
transactions = spark.read.csv('./dataset/transactions.csv', header=True, schema = schema_trans)

In [5]:
from pyspark.sql.functions import when, col, filter, to_date
b_behavior_df = b_behavior_df.withColumn('event_level', when(col('event_type').isin(["AP", "ATC", "CO"]), 'L1'). 
                         when(col('event_type').isin(["VC", "VP", "VI", "SER"]), 'L2').
                         when(col('event_type').isin(["SCR", "HP", "CL"]), 'L3'))

In [6]:
b_behavior_df.show()

+--------------------+----------+--------------------+--------------+-----------+-----------+
|          session_id|event_type|          event_time|traffic_source|device_type|event_level|
+--------------------+----------+--------------------+--------------+-----------+-----------+
|c9718135-8134-42b...|        AP|2020-12-17 07:04:...|        MOBILE|    Android|         L1|
|c9718135-8134-42b...|        CL|2020-12-06 09:07:...|        MOBILE|    Android|         L3|
|c9718135-8134-42b...|       SER|2020-12-17 07:12:...|        MOBILE|    Android|         L2|
|c9718135-8134-42b...|        CL|2021-01-08 02:28:...|        MOBILE|    Android|         L3|
|c9718135-8134-42b...|       ATC|2021-01-19 00:17:...|        MOBILE|    Android|         L1|
|c9718135-8134-42b...|        VI|2020-12-28 04:49:...|        MOBILE|    Android|         L2|
|d2077615-9026-4c5...|        HP|2020-02-23 23:03:...|        MOBILE|    Android|         L3|
|d2077615-9026-4c5...|       ATC|2020-02-24 08:56:...|      

In [10]:
from pyspark.sql import functions as F

session_features = (
    b_behavior_df.groupBy("session_id")
    .agg(
        F.sum(F.when(F.col("event_level") == "L1", 1).otherwise(0)).cast("double").alias("L1_count"),
        F.sum(F.when(F.col("event_level") == "L2", 1).otherwise(0)).cast("double").alias("L2_count"),
        F.sum(F.when(F.col("event_level") == "L3", 1).otherwise(0)).cast("double").alias("L3_count"),
    )
    .fillna(0)
)

In [32]:
b_behavior_df = b_behavior_df.withColumn("date", to_date("event_time"))

In [35]:
from pyspark.sql.functions import mean

In [36]:
b_behavior_df.groupBy('session_id').agg(mean('event_time')).show()

+--------------------+---------------+
|          session_id|avg(event_time)|
+--------------------+---------------+
|2be2c406-4fe3-455...|           NULL|
|5e2f9306-77d9-4ee...|           NULL|
|efb5e1c0-f66e-49e...|           NULL|
|79fb5117-34c6-434...|           NULL|
|8674e10b-271b-4be...|           NULL|
|6859c444-7d72-4a0...|           NULL|
|4f0e460f-b8b2-436...|           NULL|
|721a63c1-90a9-4e5...|           NULL|
|41289ace-60e5-436...|           NULL|
|c12fa8b4-e71e-413...|           NULL|
|e2e4fcd3-da31-4ca...|           NULL|
|3f165acd-8646-49c...|           NULL|
|337fc58a-b6fb-41b...|           NULL|
|dd292c5e-5ed4-434...|           NULL|
|71a60900-56f5-431...|           NULL|
|56ead6e0-072a-41f...|           NULL|
|0860887f-7154-492...|           NULL|
|16480c27-7223-46b...|           NULL|
|d0f70226-af9f-44b...|           NULL|
|1e19b0cd-b634-4b1...|           NULL|
+--------------------+---------------+
only showing top 20 rows



In [12]:
session_features = (session_features.withColumn('L1_ratio', col('L1_count')*100/(col('L1_count')+col('L2_count')+col('L3_count')))
                    .withColumn('L2_ratio', col('L2_count')*100/(col('L1_count')+col('L2_count')+col('L3_count'))))

In [ ]:
b_behavior_df

Row(session_id='2be2c406-4fe3-455a-9c12-14d7286f218c', L1_count=2, L2_count=0, L3_count=5, L1_ratio=28.571428571428573, L2_ratio=0.0)